## From dihedrals, reconstruct time evolution of the backbone


Side chains positions are missing, but at least we have something we can look at!


Output: 

- backbone_reconstruction.dcd
- backbone_reconstruction_first_frame.pdb


you can open them with VMD.

In [26]:
data_folder = "./data/"
stride = 100
dihedral_timeseries = np.loadtxt(data_folder + "hp35.dihs", delimiter = " ", dtype = float) # degrees
dihedral_movie = dihedral_timeseries[::stride] # subsample

phi_series = dihedral_movie[:, 0::2]
psi_series = dihedral_movie[:, 1::2]


print("Full dihedral array:", dihedral_timeseries.shape)
print("Subsampled dihedral array:", dihedral_movie.shape)
print("phi_series:", phi_series.shape)
print("psi_series:", psi_series.shape)

Full dihedral array: (1526041, 66)
Subsampled dihedral array: (15261, 66)
phi_series: (15261, 33)
psi_series: (15261, 33)


In [27]:
import numpy as np
import mdtraj as md
from tqdm import tqdm


# ------------------------------------------------------------
# Geometry helpers
# ------------------------------------------------------------

def place_atom(a, b, c, length, angle_deg, dihedral_deg):
    """
    Place new atom d given previous atoms a,b,c.

    Geometry:
        |c-d| = length
        angle(b-c-d) = angle_deg
        dihedral(a-b-c-d) = dihedral_deg

    Coordinates in Å.
    """

    theta = np.deg2rad(angle_deg)
    phi = np.deg2rad(dihedral_deg)

    bc = c - b
    bc /= np.linalg.norm(bc)

    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)

    m = np.cross(n, bc)

    d = (
        c
        + length * (
            -np.cos(theta) * bc
            + np.sin(theta) * (np.cos(phi) * m + np.sin(phi) * n)
        )
    )

    return d


def place_oxygen(N, CA, C):
    """
    Approximate backbone carbonyl oxygen position.
    Uses fixed C=O bond length and approximate geometry.
    """

    # Approximate peptide geometry
    C_O = 1.229
    CA_C_O_angle = 120.8

    # Put oxygen roughly trans to N around the CA-C bond
    return place_atom(N, CA, C, C_O, CA_C_O_angle, 180.0)


# ------------------------------------------------------------
# Backbone reconstruction
# ------------------------------------------------------------

def reconstruct_backbone(phi, psi, omega=None, n_residues=35):
    """
    Reconstruct N, CA, C, O backbone atoms from torsions.

    Parameters
    ----------
    phi : array, shape (n_frames, n_residues-2)
        Phi angles in degrees for residues 2,...,n_residues-1.

    psi : array, shape (n_frames, n_residues-2)
        Psi angles in degrees for residues 2,...,n_residues-1.

    omega : array, optional
        Omega angles in degrees for residues 2,...,n_residues-1.
        If None, omega = 180 degrees.

    Returns
    -------
    xyz : array, shape (n_frames, n_residues*4, 3)
        Coordinates in Å.
        Atom order per residue: N, CA, C, O.
    """

    n_frames = phi.shape[0]

    if omega is None:
        omega = 180.0 * np.ones_like(phi)

    # Standard peptide geometry in Å and degrees
    N_CA = 1.458
    CA_C = 1.525
    C_N = 1.329

    C_N_CA = 121.7
    N_CA_C = 111.2
    CA_C_N = 116.2

    xyz = np.zeros((n_frames, n_residues * 4, 3))

    for t in tqdm(range(n_frames), desc="Reconstructing backbone"):

        atoms = []

        # ----------------------------------------------------
        # Initialize first residue in a fixed reference frame
        # ----------------------------------------------------

        N0 = np.array([0.0, 0.0, 0.0])
        CA0 = np.array([N_CA, 0.0, 0.0])

        angle = np.deg2rad(N_CA_C)
        C0 = CA0 + CA_C * np.array([
            np.cos(np.pi - angle),
            np.sin(np.pi - angle),
            0.0
        ])

        O0 = place_oxygen(N0, CA0, C0)

        atoms.extend([N0, CA0, C0, O0])

        N_prev = N0
        CA_prev = CA0
        C_prev = C0

        # ----------------------------------------------------
        # Build residues 2,...,n_residues
        # ----------------------------------------------------

        for i in range(1, n_residues):

            # Map residue index to torsion column.
            #
            # Your arrays contain residues 2,...,34 only.
            # Python residue index:
            #   i = 0 -> residue 1
            #   i = 1 -> residue 2
            #   ...
            #   i = 33 -> residue 34
            #   i = 34 -> residue 35
            #
            # column = i - 1 for residues 2,...,34.

            if 1 <= i <= n_residues - 2:
                col = i - 1
                phi_i = phi[t, col]
                psi_prev = psi[t, col - 1] if col > 0 else 180.0
                omega_prev = omega[t, col - 1] if col > 0 else 180.0
            else:
                phi_i = 180.0
                psi_prev = psi[t, -1] if i == n_residues - 1 else 180.0
                omega_prev = omega[t, -1] if i == n_residues - 1 else 180.0

            # New N_i from psi_{i-1}
            N_i = place_atom(
                N_prev, CA_prev, C_prev,
                length=C_N,
                angle_deg=CA_C_N,
                dihedral_deg=psi_prev
            )

            # New CA_i from omega_{i-1}
            CA_i = place_atom(
                CA_prev, C_prev, N_i,
                length=N_CA,
                angle_deg=C_N_CA,
                dihedral_deg=omega_prev
            )

            # New C_i from phi_i
            C_i = place_atom(
                C_prev, N_i, CA_i,
                length=CA_C,
                angle_deg=N_CA_C,
                dihedral_deg=phi_i
            )

            O_i = place_oxygen(N_i, CA_i, C_i)

            atoms.extend([N_i, CA_i, C_i, O_i])

            N_prev, CA_prev, C_prev = N_i, CA_i, C_i

        xyz[t] = np.array(atoms)

    return xyz

In [28]:
# Your torsions are in degrees
# shape should be: (n_frames, 33)
print(phi_series.shape)
print(psi_series.shape)

omega_series = 180.0 * np.ones_like(phi_series)

# For testing, reconstruct only a few frames first
xyz_angstrom = reconstruct_backbone(
    phi_series,
    psi_series,
    omega_series,
    n_residues=35
)

print(xyz_angstrom.shape)

(15261, 33)
(15261, 33)


Reconstructing backbone: 100%|██████████| 15261/15261 [01:22<00:00, 185.06it/s]

(15261, 140, 3)


In [29]:
topology = md.Topology()
chain = topology.add_chain()

for i in range(35):
    residue = topology.add_residue(f"RES{i+1}", chain)

    N = topology.add_atom("N", md.element.nitrogen, residue)
    CA = topology.add_atom("CA", md.element.carbon, residue)
    C = topology.add_atom("C", md.element.carbon, residue)
    O = topology.add_atom("O", md.element.oxygen, residue)

    topology.add_bond(N, CA)
    topology.add_bond(CA, C)
    topology.add_bond(C, O)

    if i > 0:
        prev_C = list(topology.residue(i-1).atoms)[2]
        topology.add_bond(prev_C, N)

traj = md.Trajectory(
    xyz=xyz_angstrom / 10.0,  # Å -> nm
    topology=topology
)

traj.save_pdb(data_folder + "backbone_reconstruction_first_frame.pdb")
traj.save_dcd(data_folder + "backbone_reconstruction.dcd")